<a href="https://colab.research.google.com/github/Sikaaaa3/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PART 0

In [5]:
#Testing
from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")

print("API key loaded:", API_KEY is not None)

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

API key loaded: True
Client ready.



Section 1 — Talking to an LLM Programmatically

PART 1.1 - First API Call

In [6]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content

#Asking Question
answer = ask_llm("What is a loan?")
print(answer)


#Printing response for token usage
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is a loan?"},
    ],
    temperature=0.7,
    max_tokens=500,
)

print(response.usage)








A loan is an amount of money that is borrowed from a lender, such as a bank, credit union, or individual, with the agreement that it will be repaid, usually with interest. The borrower, also known as the debtor, receives the loan amount and promises to pay back the principal (the initial amount borrowed) plus interest over a specified period of time.

Here are the key components of a loan:

1. **Principal**: The initial amount borrowed.
2. **Interest**: The fee charged by the lender for borrowing the money, usually expressed as a percentage of the principal.
3. **Term**: The length of time the borrower has to repay the loan.
4. **Repayment schedule**: The frequency and amount of payments the borrower must make to repay the loan.

Loans can be used for various purposes, such as:

* Financing a home or car purchase
* Paying for education or medical expenses
* Consolidating debt
* Funding a business or startup
* Covering unexpected expenses or emergencies

There are different types of loa

Student Reasoning

PART 1.2 - Temperature: Randomness Dail

In [7]:
question = "Suggest a name for a savings product for market traders in Accra."

for temperature in [0.0, 1.2]:
    print(f"\n Temperature: {temperature} ")

    for i in range(5):
        answer = ask_llm(question, temperature=temperature)
        print(f"{i+1}. {answer}")


 Temperature: 0.0 
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Savings**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name plays on the idea of saving being a valuable treasure for market traders.
3. **Sika Saver**: "Sika" is the Ghanaian word for money, so this name incorporates a local touch.
4. **MarketMate Savings**: This name positions the savings product as a trusted companion for market traders.
5. **Kokroko Savings Plan**: "Kokroko" is a Ghanaian term for a collective savings scheme, which could appeal to market traders who are familiar with this concept.
6. **Accra Trader's Fund**: This name emphasizes the product's focus on supporting market traders in Accra.
7. **Suzyo Savings**: "Suzyo" is a Ghanaian term for "save" or "keep", which could make the product more relatable to market traders.

Choose the one that resonates the most with your target au

Student Reasoning


Section 2 — The Dataset: Loan Application Letters

In [8]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")


6 letters loaded.



Section 3 — Prompt Engineering for the Decision Support System

Part 3.1 — Component 1: Summarization

In [13]:
#Summary of L002 and L006
SUMMARY_PROMPT_V1 = "Summarise this"
V1_L002=ask_llm(f"{SUMMARY_PROMPT_V1}\n{LETTERS['L002']}")
V1_L006=ask_llm(f"{SUMMARY_PROMPT_V1}\n{LETTERS['L002']}")

#Summary using proper prompt
SUMMARY_PROMPT_V2 = """
You are an assistant to a microfinance loan officer.
Be factual and neutral.
Use only information provided in the letter.
Do not invent or assume any details.
Summarize the loan application in 3-4 sentences.
"""
V2_L002=ask_llm(f"{SUMMARY_PROMPT_V2}\n{LETTERS['L002']}", temperature=0)
V2_L006=ask_llm(f"{SUMMARY_PROMPT_V2}\n{LETTERS['L006']}", temperature=0)

print("v1")
print("L002:")
print(V1_L002)
print("\nL006:")
print(V1_L006)

print("\nv2")
print("L002:")
print(V2_L002)
print("\nL006:")
print(V2_L006)










v1
L002:
Kwame Boateng, a commercial driver in Kumasi, needs GHS 25,000 urgently to repair his vehicle and pay debts. He promises to repay when business picks up after the festive season, but currently has no collateral to offer.

L006:
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off debts. He's experiencing slow business but expects it to improve after the festive season and is willing to repay the loan when possible, despite not having collateral.

v2
L002:
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects it to improve after the festive season. He does not currently have collateral to secure the loan, but is requesting urgent assistance with repayment to be made when possible.

L006:
Kofi has submitted a loan application for GHS 

# Comparing outputs of V1 and V2

V1 produced short and general summaries, but it made an important error by identifying L006 as Kwame instead of Kofi.V2 was more detailed and structured because it included instructions to be factual,neutral, and avoid invented details.However, V2 still described Kofi as trustworthy based on his own claim.Overall, V2 was better but still needed checking for accuracy


Student Reasoning

Part 3.2 — Component 2: Structured extraction (JSON)

In [ ]:
# Prompt definition
EXTRACT_PROMPT = """
You are an assistant extracting information from loan applications.

Return ONLY a JSON object with exactly these keys:

{
  "applicant_name": "string",
  "amount_ghs": number,
  "purpose": "string",
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": true or false,
  "repayment_months": number or null
}

Rules:
- Use only information stated in the letter.
- If a field is not stated, use null.
- Do not guess or invent information.
- Return only valid JSON.

Example:

Letter:
"My name is Ama. I need GHS 5,000 to buy a sewing machine.
I make GHS 600 profit monthly and can repay in 10 months.
I have no collateral."

JSON:
{
  "applicant_name": "Ama",
  "amount_ghs": 5000,
  "purpose": "buy a sewing machine",
  "monthly_profit_ghs": 600,
  "has_collateral_or_guarantor": false,
  "repayment_months": 10
}
"""

#Extract_fields function
import json
def extract_fields(letter_text):
  try:
    response=ask_llm(f"{EXTRACT_PROMPT}\nLetter:\n{letter_text}", temperature=0)
    response=response.strip()

    if response.startswith("````json"):
      response=response[7:]

    if response.endswith("````"):
      response=response[:-3]

    return json.loads(response.strip())
  except Exception as e:
    print("Warning: Could not parse the model response")

    return None
 #Running on all six letters
 import pandas as pd


